# NicheQKV — Multi-omics Spatial Integration with Pretrained NicheFormer + QKV Cross-Attention

Pipeline:
1. Load multi-omics spatial data (RNA + Protein/ATAC)
2. NicheFormer tokenization + pretrained Transformer encoding
3. QKV Cross-Attention Fusion (within-modality + between-modality)
4. Clustering (mclust via rpy2)
5. Evaluation & Visualization

## 1. Environment Setup

In [ ]:
!pip install scanpy anndata scikit-misc pytorch-lightning transformers safetensors rpy2

In [ ]:
import os

# Auto-detect environment: Colab (Linux) vs Local (Windows)
if os.path.exists('/usr/lib/R'):
    # Colab / Linux
    os.environ['R_HOME'] = '/usr/lib/R'
    os.environ['PATH'] = '/usr/lib/R/bin:' + os.environ.get('PATH', '')
else:
    # Windows local
    os.environ['R_HOME'] = r'C:\Program Files\R\R-4.6.1'
    os.environ['PATH'] = r"C:\rtools45\usr\bin;C:\Program Files\R\R-4.6.1\bin;C:\Program Files\R\R-4.6.1\bin\x64;" + os.environ.get('PATH', '')

# Import rpy2
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, default_converter
import rpy2.robjects.conversion as cv

cv.set_conversion(default_converter + pandas2ri.converter)

# Suppress warnings
robjects.r.options(warn=-1)

# Install mclust only if it is not already installed
robjects.r("""
if (!requireNamespace("mclust", quietly = TRUE)) {
    install.packages("mclust", repos="https://cloud.r-project.org")
}
library(mclust)
""")

n_ground_truth = annotation['ground_truth'].nunique()
print(f"Number of ground truth classes: {n_ground_truth}")

n_cluster = n_ground_truth

tool = 'mclust'  # mclust, leiden, or louvain
clustering(adata, key='SpatialGlue', add_key='SpatialGlue', n_clusters=n_cluster, method=tool, use_pca=True)

In [ ]:
# Cell 4 is not needed — mclust is already loaded in Cell 3 via rpy2 Python API.

In [ ]:
import os, random, math, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
from torch import optim
from scipy.sparse import issparse, lil_matrix, csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale
from sklearn.utils import sparsefuncs
import matplotlib.pyplot as plt
import numba

warnings.filterwarnings('ignore')
sc.settings.verbosity = 0

In [ ]:
seed = 2022
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Data Loading

In [ ]:
#@title Select dataset {display-mode: "form"}
dataset_index = 2 #@param [0, 1, 2, 3, 4, 5]

datasets = {
    0: {'name': '10x_human_lymph_node_A1', 'type': '10x', 'omics1': 'RNA', 'omics2': 'ADT',
        'url': 'https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC'},
    1: {'name': '10x_human_lymph_node_D1', 'type': '10x', 'omics1': 'RNA', 'omics2': 'ADT',
        'url': 'https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz'},
    2: {'name': 'Mouse_Brain_E11_S1', 'type': 'Spatial-epigenome-transcriptome', 'omics1': 'RNA', 'omics2': 'ATAC',
        'url': 'https://drive.google.com/drive/folders/1BzE0J5t1WJ-0WiHqyg9LFE2MUra1S81b'},
    3: {'name': 'Mouse_Brain_E13_S1', 'type': 'Spatial-epigenome-transcriptome', 'omics1': 'RNA', 'omics2': 'ATAC',
        'url': 'https://drive.google.com/drive/folders/1Vqk_G1TKxu6VvVZ9y8WJyQn1D3D1D1D1'},
    4: {'name': 'Mouse_Brain_E15_S1', 'type': 'Spatial-epigenome-transcriptome', 'omics1': 'RNA', 'omics2': 'ATAC',
        'url': 'https://drive.google.com/drive/folders/1Vqk_G1TKxu6VvVZ9y8WJyQn1D3D1D1D1'},
    5: {'name': 'Mouse_Brain_E18_S1', 'type': 'Spatial-epigenome-transcriptome', 'omics1': 'RNA', 'omics2': 'ATAC',
        'url': 'https://drive.google.com/drive/folders/1Vqk_G1TKxu6VvVZ9y8WJyQn1D3D1D1D1'},
}

selected = datasets[dataset_index]
print(f"Dataset: {selected['name']}")
print(f"Type: {selected['type']}")
print(f"Omics1: {selected['omics1']}, Omics2: {selected['omics2']}")

In [ ]:
data_dir = os.path.join('data', 'Usable_Datasets', selected['name'])
assert os.path.exists(data_dir), f"Data directory not found: {data_dir}"
print(f"Data directory: {data_dir}")
print(os.listdir(data_dir))

In [ ]:
adata_RNA = sc.read_h5ad(os.path.join(data_dir, 'adata_RNA.h5ad'))
if selected['type'] == '10x':
    adata_omics2 = sc.read_h5ad(os.path.join(data_dir, 'adata_ADT.h5ad'))
else:
    adata_omics2 = sc.read_h5ad(os.path.join(data_dir, 'adata_ATAC.h5ad'))

# Handle different annotation filenames and formats
anno_path_csv = os.path.join(data_dir, 'annotation.csv')
anno_path_anno = os.path.join(data_dir, 'anno.csv')
if os.path.exists(anno_path_csv):
    annotation = pd.read_csv(anno_path_csv, index_col=0)
elif os.path.exists(anno_path_anno):
    annotation = pd.read_csv(anno_path_anno, index_col=0)
else:
    raise FileNotFoundError(f"No annotation file found in {data_dir}")

# Normalize column name to 'ground_truth'
annotation.columns = ['ground_truth'] if annotation.shape[1] == 1 else annotation.columns
if annotation.shape[1] > 1:
    # Mouse Brain has columns [0, cluster] — use 'cluster' (last col with names)
    annotation = annotation.iloc[:, -1:].copy()
    annotation.columns = ['ground_truth']

print(f"Annotation columns: {annotation.columns.tolist()}")
print(f"Sample labels: {annotation['ground_truth'].unique()[:5]}")

common_cells = adata_RNA.obs_names.intersection(adata_omics2.obs_names)
adata_RNA = adata_RNA[common_cells].copy()
adata_omics2 = adata_omics2[common_cells].copy()
annotation = annotation.loc[annotation.index.isin(common_cells)].copy()

print(f"RNA: {adata_RNA.shape}")
print(f"Omics2: {adata_omics2.shape}")
print(f"Annotation: {annotation.shape}")
print(f"Ground-truth labels: {annotation['ground_truth'].nunique()} unique")

## 3. Preprocessing

In [ ]:
sc.pp.filter_genes(adata_RNA, min_cells=10)
if selected['type'] == 'Spatial-epigenome-transcriptome':
    sc.pp.filter_cells(adata_RNA, min_genes=200)
    common_cells = adata_RNA.obs_names.intersection(adata_omics2.obs_names)
    adata_RNA = adata_RNA[common_cells].copy()
    adata_omics2 = adata_omics2[common_cells].copy()

print(f"RNA after filtering: {adata_RNA.shape}")

### 3b. Omics2 Preprocessing

In [ ]:
def clr_normalize_each_cell(adata, inplace=True):
    def seurat_clr(x):
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x))
        return np.log1p(x / exp)
    if not inplace:
        adata = adata.copy()
    adata.X = np.apply_along_axis(
        seurat_clr, 1,
        (adata.X.toarray() if issparse(adata.X) else np.array(adata.X))
    )
    return adata

def tfidf(X):
    idf = X.shape[0] / X.sum(axis=0)
    if issparse(X):
        tf = X.multiply(1 / X.sum(axis=1))
        return tf.multiply(idf)
    else:
        tf = X / X.sum(axis=1, keepdims=True)
        return tf * idf

def lsi(adata, n_components=20, use_highly_variable=None):
    if use_highly_variable is None:
        use_highly_variable = "highly_variable" in adata.var
    adata_use = adata[:, adata.var["highly_variable"]] if use_highly_variable else adata
    X = tfidf(adata_use.X)
    X_norm = sklearn_preprocessing_norm(X)
    X_norm = np.log1p(X_norm * 1e4)
    from sklearn.utils.extmath import randomized_svd
    X_lsi = randomized_svd(X_norm, n_components)[0]
    X_lsi -= X_lsi.mean(axis=1, keepdims=True)
    X_lsi /= X_lsi.std(axis=1, ddof=1, keepdims=True)
    adata.obsm["X_lsi"] = X_lsi[:, 1:]

def sklearn_preprocessing_norm(X):
    from sklearn.preprocessing import normalize
    if issparse(X):
        return normalize(X, norm='l1')
    else:
        return X / X.sum(axis=1, keepdims=True)

In [ ]:
if selected['type'] == '10x':
    adata_omics2 = clr_normalize_each_cell(adata_omics2)
    adata_omics2.X = scale(adata_omics2.X)
    pca = PCA(n_components=min(50, adata_omics2.X.shape[1]-1), random_state=seed)
    adata_omics2.obsm['feat'] = pca.fit_transform(adata_omics2.X)
else:
    if 'highly_variable' not in adata_omics2.var.columns:
        sc.pp.highly_variable_genes(adata_omics2, n_top_genes=2000, flavor='seurat_v3')
    lsi(adata_omics2, n_components=20)
    adata_omics2.obsm['feat'] = adata_omics2.obsm['X_lsi']

print(f"Omics2 features: {adata_omics2.obsm['feat'].shape}")

## 4. Graph Construction

In [ ]:
def build_spatial_graph(coords, n_neighbors=6):
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean').fit(coords)
    _, indices = nbrs.kneighbors(coords)
    n = coords.shape[0]
    A = lil_matrix((n, n))
    for i in range(n):
        for j in indices[i]:
            A[i, j] = 1
    A = A + A.T
    A[A > 1] = 1
    A = A.tocsr()
    # Symmetric degree normalization using sparse.diags (not np.diag)
    D = np.array(A.sum(axis=1)).flatten()
    D_inv_sqrt = np.power(D, -0.5)
    D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
    D_inv_sqrt_mat = csr_matrix(np.diag(D_inv_sqrt))
    A_norm = D_inv_sqrt_mat @ A @ D_inv_sqrt_mat
    return A_norm

def build_feature_graph(features, k=20):
    nbrs = NearestNeighbors(n_neighbors=k, metric='correlation').fit(features)
    _, indices = nbrs.kneighbors(features)
    n = features.shape[0]
    A = lil_matrix((n, n))
    for i in range(n):
        for j in indices[i]:
            A[i, j] = 1
    A = A + A.T
    A[A > 1] = 1
    A = A.tocsr()
    D = np.array(A.sum(axis=1)).flatten()
    D_inv_sqrt = np.power(D, -0.5)
    D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
    D_inv_sqrt_mat = csr_matrix(np.diag(D_inv_sqrt))
    A_norm = D_inv_sqrt_mat @ A @ D_inv_sqrt_mat
    return A_norm

def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse_coo_tensor(indices, values, shape)

In [ ]:
coords = adata_RNA.obsm['spatial'] if 'spatial' in adata_RNA.obsm else adata_RNA.obs[['x', 'y']].values
n_neighbors = 3 if selected['type'] == '10x' else 6
adj_spatial = build_spatial_graph(coords, n_neighbors=n_neighbors)
print(f"Spatial graph: {adj_spatial.shape}, nnz={adj_spatial.nnz}")

## 5. NicheFormer Tokenization

Gene-ranking-based tokenization from NicheFormer:
- `sf_normalize`: scale each cell's counts to sum=10000
- Median normalization: divide by per-gene median
- Rank genes by normalized expression, take top 1500
- Offset by `aux_tokens=30`

In [ ]:
def sf_normalize(X):
    X = X.copy()
    counts = np.array(X.sum(axis=1))
    counts += counts == 0.
    scaling_factor = 10000. / counts
    if issparse(X):
        sparsefuncs.inplace_row_scale(X, scaling_factor)
    else:
        np.multiply(X, scaling_factor.reshape((-1, 1)), out=X)
    return X

@numba.jit(nopython=True, nogil=True)
def _sub_tokenize_data(x, max_seq_len=-1, aux_tokens=30):
    scores_final = np.empty((x.shape[0], max_seq_len if max_seq_len > 0 else x.shape[1]))
    for i, cell in enumerate(x):
        nonzero_mask = np.nonzero(cell)[0]
        sorted_indices = nonzero_mask[np.argsort(-cell[nonzero_mask])][:max_seq_len]
        sorted_indices = sorted_indices + aux_tokens
        if max_seq_len:
            scores = np.zeros(max_seq_len, dtype=np.int32)
        else:
            scores = np.zeros_like(cell, dtype=np.int32)
        scores[:len(sorted_indices)] = sorted_indices.astype(np.int32)
        scores_final[i, :] = scores
    return scores_final

def tokenize_for_nicheformer(X_raw, max_seq_len=1500, aux_tokens=30):
    X = X_raw.copy()
    if issparse(X):
        X = X.toarray()
    X = np.nan_to_num(X)
    X = sf_normalize(X)
    median_counts_per_gene = np.median(X, axis=0)
    median_counts_per_gene += median_counts_per_gene == 0
    X = X / median_counts_per_gene.reshape((1, -1))
    tokens = _sub_tokenize_data(X, max_seq_len, aux_tokens).astype(np.int32)
    return tokens

In [ ]:
rna_tokens = tokenize_for_nicheformer(adata_RNA.X, max_seq_len=1500)
print(f"RNA tokens: {rna_tokens.shape}")

## 6. Load Pretrained NicheFormer

Architecture: 12-layer Transformer encoder, 16 heads, 512-dim model.
Pre-trained with Masked Language Model (MLM) on ~110M cells.

Loaded from HuggingFace: `theislab/Nicheformer`

In [ ]:
from transformers import AutoModel

nicheformer_model = AutoModel.from_pretrained(
    "theislab/Nicheformer",
    trust_remote_code=True
)
nicheformer_model = nicheformer_model.to(device)
nicheformer_model.eval()

print(f"Model loaded: {sum(p.numel() for p in nicheformer_model.parameters()):,} parameters")
print(f"Model type: {type(nicheformer_model)}")

### 6b. Extract NicheFormer Embeddings

In [ ]:
def extract_nicheformer_embeddings(model, tokens_np, batch_size=16, layer=-1):
    """
    Extract embeddings from the pretrained NicheFormer.
    Processes tokens in batches, drops context tokens (first 3), mean-pools over gene tokens.
    """
    model.eval()
    all_embs = []
    n_cells = tokens_np.shape[0]
    padding_token = 1

    with torch.no_grad():
        for start in range(0, n_cells, batch_size):
            end = min(start + batch_size, n_cells)
            batch_tokens = torch.LongTensor(tokens_np[start:end]).to(next(model.parameters()).device)

            # Replace padding token 0 -> 1
            batch_tokens = torch.where(batch_tokens == 0, torch.tensor(padding_token, device=batch_tokens.device), batch_tokens)
            attention_mask = (batch_tokens == padding_token)
            attention_mask = attention_mask.bool()

            # Try HuggingFace-style forward pass first
            try:
                outputs = model(
                    input_ids=batch_tokens,
                    attention_mask=attention_mask
                )
                # Handle different output formats
                if hasattr(outputs, 'last_hidden_state'):
                    hidden = outputs.last_hidden_state
                elif hasattr(outputs, 'hidden_states') and outputs.hidden_states is not None:
                    hidden = outputs.hidden_states[layer]
                elif isinstance(outputs, dict):
                    hidden = outputs.get('last_hidden_state', outputs.get('logits', list(outputs.values())[0]))
                else:
                    hidden = outputs[0] if isinstance(outputs, tuple) else outputs
            except TypeError:
                # Fallback: call model.forward directly with positional args
                token_embedding = model.embeddings(batch_tokens)
                if hasattr(model, 'positional_embedding'):
                    pos = torch.arange(0, batch_tokens.shape[1], dtype=torch.long, device=batch_tokens.device)
                    pos_embedding = model.positional_embedding(pos)
                    embeddings = model.dropout(token_embedding + pos_embedding) if hasattr(model, 'dropout') else (token_embedding + pos_embedding)
                else:
                    embeddings = token_embedding
                if hasattr(model.encoder, 'layers'):
                    layer_idx = len(model.encoder.layers) + layer if layer < 0 else layer
                    for i in range(layer_idx + 1):
                        embeddings = model.encoder.layers[i](
                            embeddings, src_key_padding_mask=attention_mask, is_causal=False
                        )
                    hidden = embeddings

            # Drop first 3 context tokens, mean-pool over gene tokens
            hidden = hidden[:, 3:, :]
            hidden = hidden.mean(dim=1)
            all_embs.append(hidden.cpu())
        del batch_tokens, outputs, hidden
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return torch.cat(all_embs, dim=0)

In [ ]:
emb_rna = extract_nicheformer_embeddings(nicheformer_model, rna_tokens, batch_size=16)
emb_rna = emb_rna.numpy()
print(f"RNA embeddings (NicheFormer): {emb_rna.shape}")

In [ ]:
feat2 = adata_omics2.obsm['feat']
pca_512 = PCA(n_components=min(512, feat2.shape[1]), random_state=seed)
emb_omics2 = pca_512.fit_transform(feat2)
if emb_omics2.shape[1] < 512:
    pad = np.zeros((emb_omics2.shape[0], 512 - emb_omics2.shape[1]))
    emb_omics2 = np.hstack([emb_omics2, pad])
print(f"Omics2 embeddings: {emb_omics2.shape}")

## 7. Feature Graph Construction (built after embedding extraction)

In [ ]:
adj_feature1 = build_feature_graph(emb_rna, k=20)
adj_feature2 = build_feature_graph(emb_omics2, k=20)
print(f"Feature graph RNA: {adj_feature1.shape}")
print(f"Feature graph Omics2: {adj_feature2.shape}")

## 8. QKV Cross-Attention Fusion Model

Architecture:
1. GCN encoder per modality x graph type
2. QKV Cross-Attention within-modality fusion (spatial + feature)
3. QKV Cross-Attention between-modality fusion
4. GCN decoders for reconstruction loss
5. Consistency encoding (cross-reconstruction)

In [ ]:
class QKVCrossFusionLayer(nn.Module):
    def __init__(self, dim, attention_type='local'):
        super().__init__()
        self.dim = dim
        self.attention_type = attention_type
        self.scale = dim ** -0.5

        self.q_proj1 = nn.Linear(dim, dim, bias=False)
        self.k_proj1 = nn.Linear(dim, dim, bias=False)
        self.v_proj1 = nn.Linear(dim, dim, bias=False)

        self.q_proj2 = nn.Linear(dim, dim, bias=False)
        self.k_proj2 = nn.Linear(dim, dim, bias=False)
        self.v_proj2 = nn.Linear(dim, dim, bias=False)

        self.fc_out = nn.Linear(2 * dim, dim, bias=True)

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.q_proj1.weight)
        nn.init.xavier_uniform_(self.k_proj1.weight)
        nn.init.xavier_uniform_(self.v_proj1.weight)
        nn.init.xavier_uniform_(self.q_proj2.weight)
        nn.init.xavier_uniform_(self.k_proj2.weight)
        nn.init.xavier_uniform_(self.v_proj2.weight)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, emb1, emb2):
        q1 = self.q_proj1(emb1)
        k1 = self.k_proj1(emb1)
        v1 = self.v_proj1(emb1)

        q2 = self.q_proj2(emb2)
        k2 = self.k_proj2(emb2)
        v2 = self.v_proj2(emb2)

        if self.attention_type == 'global':
            attn_scores1 = torch.matmul(q2, k1.T) * self.scale
            attn_probs1 = F.softmax(attn_scores1, dim=-1)
            z1 = torch.matmul(attn_probs1, v1)

            attn_scores2 = torch.matmul(q1, k2.T) * self.scale
            attn_probs2 = F.softmax(attn_scores2, dim=-1)
            z2 = torch.matmul(attn_probs2, v2)

            alpha1 = attn_probs1.mean(dim=-1, keepdim=True)
            alpha2 = attn_probs2.mean(dim=-1, keepdim=True)
            alpha = torch.cat([alpha1, alpha2], dim=-1)
        else:
            score1 = (q2 * k1).sum(dim=-1, keepdim=True) * self.scale
            score2 = (q1 * k2).sum(dim=-1, keepdim=True) * self.scale

            weight1 = torch.sigmoid(score1)
            weight2 = torch.sigmoid(score2)

            z1 = weight1 * v1
            z2 = weight2 * v2

            alpha = torch.cat([weight1, weight2], dim=-1)

        z_concat = torch.cat([z1, z2], dim=-1)
        fused = self.fc_out(z_concat)

        return fused, alpha


class GCNEncoder(nn.Module):
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_feat, out_feat))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, feat, adj):
        x = torch.mm(feat, self.weight)
        x = torch.spmm(adj, x)
        return x


class GCNDecoder(nn.Module):
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_feat, out_feat))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, feat, adj):
        x = torch.mm(feat, self.weight)
        x = torch.spmm(adj, x)
        return x


class NicheQKVModel(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=64, attention_type='local'):
        super().__init__()
        self.encoder_omics1 = GCNEncoder(input_dim, hidden_dim)
        self.decoder_omics1 = GCNDecoder(hidden_dim, input_dim)
        self.encoder_omics2 = GCNEncoder(input_dim, hidden_dim)
        self.decoder_omics2 = GCNDecoder(hidden_dim, input_dim)

        self.atten_omics1 = QKVCrossFusionLayer(hidden_dim, attention_type=attention_type)
        self.atten_omics2 = QKVCrossFusionLayer(hidden_dim, attention_type=attention_type)
        self.atten_cross = QKVCrossFusionLayer(hidden_dim, attention_type=attention_type)

    def forward(self, feat1, feat2, adj_sp1, adj_ft1, adj_sp2, adj_ft2):
        emb_sp1 = self.encoder_omics1(feat1, adj_sp1)
        emb_ft1 = self.encoder_omics1(feat1, adj_ft1)
        emb_sp2 = self.encoder_omics2(feat2, adj_sp2)
        emb_ft2 = self.encoder_omics2(feat2, adj_ft2)

        emb_fused1, alpha1 = self.atten_omics1(emb_sp1, emb_ft1)
        emb_fused2, alpha2 = self.atten_omics2(emb_sp2, emb_ft2)

        emb_combined, alpha_cross = self.atten_cross(emb_fused1, emb_fused2)

        recon1 = self.decoder_omics1(emb_combined, adj_sp1)
        recon2 = self.decoder_omics2(emb_combined, adj_sp2)

        emb1_cross = self.encoder_omics2(self.decoder_omics2(emb_fused1, adj_sp2), adj_sp2)
        emb2_cross = self.encoder_omics1(self.decoder_omics1(emb_fused2, adj_sp1), adj_sp1)

        return {
            'emb_latent_omics1': emb_fused1,
            'emb_latent_omics2': emb_fused2,
            'emb_latent_combined': emb_combined,
            'emb_recon_omics1': recon1,
            'emb_recon_omics2': recon2,
            'emb_latent_omics1_across_recon': emb1_cross,
            'emb_latent_omics2_across_recon': emb2_cross,
            'alpha_omics1': alpha1,
            'alpha_omics2': alpha2,
            'alpha': alpha_cross,
        }

### 8b. Training

In [ ]:
def preprocess_graph(adj):
    """Add self-loops + symmetric normalization, return torch sparse tensor."""
    adj = adj.tocoo()
    adj_ = adj + csr_matrix(np.eye(adj.shape[0]))
    rowsum = np.array(adj_.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0
    D_inv_sqrt = csr_matrix(np.diag(d_inv_sqrt))
    adj_norm = D_inv_sqrt @ adj_ @ D_inv_sqrt
    return sparse_mx_to_torch_sparse_tensor(adj_norm)

adj_sp1 = preprocess_graph(adj_spatial).to(device)
adj_sp2 = preprocess_graph(adj_spatial).to(device)
adj_ft1 = preprocess_graph(adj_feature1).to(device)
adj_ft2 = preprocess_graph(adj_feature2).to(device)

features_omics1 = torch.FloatTensor(emb_rna).to(device)
features_omics2 = torch.FloatTensor(emb_omics2).to(device)

print(f"Feature tensors: omics1={features_omics1.shape}, omics2={features_omics2.shape}")

In [ ]:
if selected['type'] == '10x':
    n_epochs = 200
    weight_factors = [1, 5, 1, 10]
elif selected['type'] == 'SPOTS':
    n_epochs = 600
    weight_factors = [1, 5, 1, 1]
elif 'Stereo' in selected['type']:
    n_epochs = 1500
    weight_factors = [1, 10, 1, 10]
else:
    n_epochs = 1600
    weight_factors = [1, 5, 1, 1]

dim_input1 = features_omics1.shape[1]
dim_input2 = features_omics2.shape[1]
dim_output = 64

model = NicheQKVModel(
    input_dim=dim_output,
    hidden_dim=dim_output,
    attention_type='local'
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.00)

# Project input features to hidden dim first
proj1 = nn.Linear(dim_input1, dim_output).to(device)
proj2 = nn.Linear(dim_input2, dim_output).to(device)
proj1_bn = nn.BatchNorm1d(dim_output).to(device)
proj2_bn = nn.BatchNorm1d(dim_output).to(device)
proj_params = list(proj1.parameters()) + list(proj2.parameters()) + list(proj1_bn.parameters()) + list(proj2_bn.parameters())
optimizer = torch.optim.Adam(list(model.parameters()) + proj_params, lr=0.0001, weight_decay=0.00)

print(f"Epochs: {n_epochs}, Weight factors: {weight_factors}")
print(f"Training model with input dims: {dim_input1}, {dim_input2} -> hidden: {dim_output}")

In [ ]:
model.train()
for epoch in range(n_epochs):
    optimizer.zero_grad()

    feat1_proj = proj1_bn(proj1(features_omics1))
    feat2_proj = proj2_bn(proj2(features_omics2))

    results = model(feat1_proj, feat2_proj, adj_sp1, adj_ft1, adj_sp2, adj_ft2)

    loss_recon1 = F.mse_loss(results['emb_recon_omics1'], feat1_proj)
    loss_recon2 = F.mse_loss(results['emb_recon_omics2'], feat2_proj)
    loss_corr1 = F.mse_loss(results['emb_latent_omics1'], results['emb_latent_omics1_across_recon'])
    loss_corr2 = F.mse_loss(results['emb_latent_omics2'], results['emb_latent_omics2_across_recon'])

    loss = (weight_factors[0] * loss_recon1 + weight_factors[1] * loss_recon2 +
            weight_factors[2] * loss_corr1 + weight_factors[3] * loss_corr2)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{n_epochs}  Loss: {loss.item():.4f}  "
              f"(recon1={loss_recon1.item():.4f}, recon2={loss_recon2.item():.4f}, "
              f"corr1={loss_corr1.item():.4f}, corr2={loss_corr2.item():.4f})")

print(f"\nModel training finished! Final Loss: {loss.item():.4f}")

### 8c. Extract Final Embeddings

In [ ]:
model.eval()
with torch.no_grad():
    feat1_proj = proj1_bn(proj1(features_omics1))
    feat2_proj = proj2_bn(proj2(features_omics2))
    results = model(feat1_proj, feat2_proj, adj_sp1, adj_ft1, adj_sp2, adj_ft2)

    emb_combined = F.normalize(results['emb_latent_combined'], p=2, eps=1e-12, dim=1)
    emb_combined = emb_combined.cpu().numpy()

adata = adata_RNA.copy()
adata.obsm['emb_latent_omics1'] = F.normalize(results['emb_latent_omics1'], p=2, eps=1e-12, dim=1).cpu().numpy()
adata.obsm['emb_latent_omics2'] = F.normalize(results['emb_latent_omics2'], p=2, eps=1e-12, dim=1).cpu().numpy()
adata.obsm['SpatialGlue'] = emb_combined
adata.obsm['alpha'] = results['alpha'].cpu().numpy()
adata.obsm['alpha_omics1'] = results['alpha_omics1'].cpu().numpy()
adata.obsm['alpha_omics2'] = results['alpha_omics2'].cpu().numpy()

print(f"Final embedding: {emb_combined.shape}")

## 9. Clustering (mclust via rpy2)

In [ ]:
n_clusters = annotation['ground_truth'].nunique()
print(f"Number of clusters: {n_clusters}")

In [ ]:
pca_20 = PCA(n_components=20, random_state=seed)
emb_pca = pca_20.fit_transform(emb_combined)
adata_RNA.obsm['pca_for_clustering'] = emb_pca

In [ ]:
import numpy as np
import rpy2.robjects as robjects
from rpy2.robjects import numpy2ri
import rpy2.robjects.conversion as cv

# Activate numpy conversion
numpy2ri.activate()

rmclust = robjects.r["Mclust"]

# Convert Python variables to R
r_emb_pca = robjects.r["as.matrix"](robjects.FloatVector(emb_pca.flatten().tolist()))
r_emb_pca = robjects.r["matrix"](r_emb_pca, nrow=emb_pca.shape[0], ncol=emb_pca.shape[1])
r_n_clusters = robjects.IntVector([n_clusters])

# Run Mclust
mc = rmclust(r_emb_pca, G=r_n_clusters, modelNames="EEE", initialization="subset", subset=300)

# Extract classification back to Python
cluster_labels = np.array(mc.rx2["classification"])

# Deactivate numpy conversion
numpy2ri.deactivate()

print(f"Mclust completed: {len(cluster_labels)} cells, {len(np.unique(cluster_labels))} clusters")

In [ ]:
adata_RNA.obs['NicheQKV'] = pd.Categorical(cluster_labels.astype(str))
print(f"Clusters assigned: {adata_RNA.obs['NicheQKV'].nunique()} unique")

## 10. Evaluation Metrics

In [ ]:
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             adjusted_mutual_info_score, homogeneity_score,
                             v_measure_score, silhouette_score)

true_labels = annotation['ground_truth'].astype(str).values
pred_labels = np.array(adata_RNA.obs['NicheQKV'].values, dtype=str)

ari = adjusted_rand_score(true_labels, pred_labels)
nmi = normalized_mutual_info_score(true_labels, pred_labels)
ami = adjusted_mutual_info_score(true_labels, pred_labels)
homogeneity = homogeneity_score(true_labels, pred_labels)
v_measure = v_measure_score(true_labels, pred_labels)
silhouette = silhouette_score(emb_combined, pred_labels)

print(f"ARI:          {ari:.4f}")
print(f"NMI:          {nmi:.4f}")
print(f"AMI:          {ami:.4f}")
print(f"Homogeneity:  {homogeneity:.4f}")
print(f"V-measure:    {v_measure:.4f}")
print(f"Silhouette:   {silhouette:.4f}")

## 11. Visualization

In [ ]:
sc.pp.neighbors(adata_RNA, use_rep='SpatialGlue')
sc.tl.umap(adata_RNA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sc.pl.umap(adata_RNA, color='NicheQKV', ax=axes[0], show=False, title='NicheQKV Clusters')
adata_RNA.obs['Ground Truth'] = annotation['ground_truth'].astype(str).values
sc.pl.umap(adata_RNA, color='Ground Truth', ax=axes[1], show=False, title='Ground Truth')
adata_RNA.obs['Modality'] = selected['type']
sc.pl.umap(adata_RNA, color='Modality', ax=axes[2], show=False, title='Modality')
plt.tight_layout()
plt.show()

## Summary

**Pipeline:**
1. Raw RNA + Omics2 loaded -> basic filtering (raw counts preserved)
2. NicheFormer tokenization (sf_normalize -> median scaling -> gene ranking -> top 1500 tokens)
3. Pretrained NicheFormer (12-layer Transformer, 512-dim) -> RNA embeddings
4. Omics2 projected to 512-dim via PCA
5. Spatial graph (KNN on coords) + Feature graphs (KNN on embeddings)
6. QKV Cross-Attention Fusion: GCN encode -> within-modality -> between-modality
7. Reconstruction + consistency losses
8. mclust clustering -> evaluation